In [2]:
!pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [kagglehub]/5 [kagglesdk]


In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ffatty/plain-text-wikipedia-simpleenglish")

print("Path to dataset files:", path)

100%|██████████| 128M/128M [00:01<00:00, 84.0MB/s] 

Extracting files...


Path to dataset files: /home/sraja/.cache/kagglehub/datasets/ffatty/plain-text-wikipedia-simpleenglish/versions/2


In [17]:
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.6/800.6 kB 52.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [nltk]2/3 [nltk]


In [18]:
#Load txt file from the dataset
import urllib.request
import nltk
import numpy as np
from nltk import word_tokenize
from nltk.probability import FreqDist
from collections import defaultdict, Counter
import urllib.request
import csv

filepath = "/home/sraja/.cache/kagglehub/datasets/ffatty/plain-text-wikipedia-simpleenglish/versions/2/AllCombined.txt"
with open(filepath, 'r', encoding='utf-8') as file:
    long_txt = file.read()

In [19]:
#Print the first 500 characters of the text
print(long_txt[:500])


April

April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.

April always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of the week as December.

April comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and No


In [24]:
# Tokenize the text
tokens = word_tokenize(long_txt.lower())

## UNIGRAM
# Build a frequency distribution of the tokens
freq_dist = nltk.FreqDist(tokens)
# Normalize the frequencies to get probabilities
total_frequency = sum(freq_dist.values())
probabilities = {word: freq/total_frequency for word, freq in freq_dist.items()}

# BIGRAM
# Compute the bigrams
bigrams = list(nltk.bigrams(tokens))
total_bigrams = float(len(bigrams))
# Compute the frequency distribution of the bigrams
bigram_freq = nltk.FreqDist(bigrams)
# Initialize a two-dimensional default dictionary. This will store our model.
bi_model = defaultdict(Counter)

# Populate the model with the bigram probabilities
# Write token probabilities to a text file
with open('bigram_probabilities.txt', 'w') as f:
    writer = csv.writer(f)
    for word1, word2 in bigrams:
        bi_model[word1][word2] = (bigram_freq[(word1, word2)] / total_bigrams)
        writer.writerow([word1, word2, bi_model[word1][word2]])

KeyboardInterrupt: 

In [ ]:
#sample a word based on the bigram probabilities
# COMBINE the two probabilities
r = 0.7
for word1 in bi_model: 
    prob_sum = sum(bi_model[word1].values())
    for word, freq in freq_dist.items():
        if word in bi_model[word1]: 
            bi_model[word1][word] = bi_model[word1][word]*r/prob_sum + freq*(1-r)/total_frequency
        else: 
            bi_model[word1][word] = freq/total_frequency

# normalize probability to add up to 1
for word1 in bi_model: 
    prob_sum = sum(bi_model[word1].values())
    for word, freq in freq_dist.items():
        bi_model[word1][word] = bi_model[word1][word]/prob_sum


[nltk_data] Downloading package punkt_tab to /home/sraja/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Sample a word from the probability distribution
def sample_word(probabilities):
    words = list(probabilities.keys())
    probs = list(probabilities.values())
    word = np.random.choice(words, p=probs)
    return word

def sample_bi_gram(bi_model, word1): 
    probabilities = bi_model[word1]
    return sample_word(probabilities)
    

In [ ]:
# Test the function
print(sample_word(probabilities))

In [ ]:
print("what", sample_bi_gram(bi_model, "what"))

In [ ]:
# Repeat word sample to generate a sequence of words
import time
import re
import string
    
# Repeat word sampling from bi-grams to generate a sentence
def generate_bigram_sentence(model, start):
    word = start
    while word not in ['.', '!', '?']:
        if re.fullmatch(r'['+re.escape(string.punctuation)+']*$', word):
            print(f"{word}", end="")
        else:
            print(f" {word}", end="")
        word = sample_bi_gram(model, word)
        time.sleep(0.5)
    print(word)

In [ ]:
generate_bigram_sentence(bi_model, "what")